# GravWaveFormer - Notebook 3: Model Architecture
**EGN 6217 Applied Deep Learning**

Defines all model architectures, tests them with dummy data,
and saves model classes for use in training and evaluation.

In [1]:
# Setup and Install
from google.colab import drive
drive.mount('/content/drive')

!pip install open-clip-torch --quiet

import os, torch, torch.nn as nn, torch.nn.functional as F
from torchvision import models
import numpy as np

PROJECT_DIR = '/content/drive/MyDrive/GravWaveFormer'
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
Device: cuda
  GPU: Tesla T4


In [2]:
#  CELL 1 — SETUP AND INSTALLS

from google.colab import drive
drive.mount('/content/drive')

# Install all required packages
# torch-geometric is for the GNN; transformers/BLIP-2 for the VLM
import subprocess
subprocess.run(["pip", "install", "-q",
    "transformers",            # HuggingFace: CLIP, BLIP-2
    "accelerate",              # Required by BLIP-2 for quantization
    "bitsandbytes",            # 8-bit quantization (fits BLIP-2 on T4)
    "torch-geometric",         # Graph Neural Networks
    "torch-scatter",           # GNN dependency
    "torch-sparse",            # GNN dependency
    "open-clip-torch",         # OpenCLIP (open-source CLIP)
    "scikit-learn",
    "pytorch-grad-cam",
])

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np

PROJECT_DIR = '/content/drive/MyDrive/GravWaveFormer'
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results',     exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")
if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  WARNING: GPU not found. Recommend Runtime → Change runtime type → GPU (T4 or A100)")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Using device: cuda
  GPU: Tesla T4
  Memory: 15.6 GB


In [3]:
#   CELL 2 — DATASET CLASS (unchanged from v1, repeated here)

import pandas as pd
from torch.utils.data import Dataset, DataLoader

class GravWaveDataset(Dataset):
    """
    Loads pre-cached spectrogram tensors (.pt files) for 2D models.
    Returns (tensor [3,224,224], label).
    augment=True applies random flips + additive noise during training.
    """
    def __init__(self, split_csv_path, spectrogram_dir, augment=False):
        self.data     = pd.read_csv(split_csv_path)
        self.spec_dir = spectrogram_dir
        self.augment  = augment
        existing      = set(f.replace('.pt', '') for f in os.listdir(spectrogram_dir))
        self.data     = self.data[self.data['id'].isin(existing)].reset_index(drop=True)
        print(f"  Dataset loaded: {len(self.data):,} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row    = self.data.iloc[idx]
        tensor = torch.load(os.path.join(self.spec_dir, f"{row['id']}.pt"))
        label  = torch.tensor(float(row['target']), dtype=torch.float32)
        if self.augment:
            if torch.rand(1).item() > 0.5:
                tensor = torch.flip(tensor, dims=[2])
            if torch.rand(1).item() > 0.5:
                tensor = torch.flip(tensor, dims=[1])
            tensor = tensor + torch.randn_like(tensor) * 0.02
        return tensor, label


class GravWaveDataset1D(Dataset):
    """
    NEW: Loads pre-cached 1D waveform tensors (.pt files) for WaveCNN1D.
    These are saved in Notebook_02 under waveforms/ directory.
    Returns (tensor [3, 4096], label) — 3 detectors × 4096 samples.
    """
    def __init__(self, split_csv_path, waveform_dir, augment=False):
        self.data      = pd.read_csv(split_csv_path)
        self.wave_dir  = waveform_dir
        self.augment   = augment
        existing       = set(f.replace('.pt', '') for f in os.listdir(waveform_dir))
        self.data      = self.data[self.data['id'].isin(existing)].reset_index(drop=True)
        print(f"  1D Dataset loaded: {len(self.data):,} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row    = self.data.iloc[idx]
        tensor = torch.load(os.path.join(self.wave_dir, f"{row['id']}.pt"))
        label  = torch.tensor(float(row['target']), dtype=torch.float32)
        if self.augment:
            # Time shift: circular roll by a random offset up to ±200 samples
            shift  = torch.randint(-200, 200, (1,)).item()
            tensor = torch.roll(tensor, shift, dims=1)
            # Amplitude jitter: scale each detector by a small random factor
            scale  = 1.0 + (torch.rand(3, 1) - 0.5) * 0.1  # ±5% per channel
            tensor = tensor * scale
            # Additive noise
            tensor = tensor + torch.randn_like(tensor) * 0.01
        return tensor, label



In [4]:
#   CELL 3 — MODEL 1: GravWaveFormer (upgraded from v1)
#
#  Changes from v1:
#  • d_model bumped from 256 → 512 for richer token representations
#  • Transformer depth increased from 4 → 6 layers
#  • Added stochastic depth (drop path) for stronger regularisation
#  • forward() now optionally returns intermediate patch embeddings
#    so the ensemble can access richer features than just CLS

class DropPath(nn.Module):
    """
    Stochastic Depth / Drop Path regularisation.
    Randomly drops entire residual paths during training.
    Stronger regulariser than standard dropout for Transformers.
    Introduced in "Deep Networks with Stochastic Depth" (Huang et al.)
    """
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob = 1 - self.drop_prob
        shape      = (x.shape[0],) + (1,) * (x.ndim - 1)
        rand_tensor = torch.rand(shape, dtype=x.dtype, device=x.device)
        rand_tensor = torch.floor(rand_tensor + keep_prob)
        return x / keep_prob * rand_tensor


class GravWaveFormer(nn.Module):
    """
    GravWave-Former: ResNet-18 CNN + deeper Transformer encoder.

    Architecture:
        Input (3, 224, 224)
            → ResNet-18 (layer4 fine-tuned)      → (B, 512, 7, 7)
            → Patch projection (512→d_model)      → (B, 49, d_model)
            → Prepend [CLS] + positional encoding → (B, 50, d_model)
            → 6-layer Transformer encoder
            → CLS output → classifier → P(GW)

    Changes from v1:
        • d_model: 256 → 512
        • num_layers: 4 → 6
        • Added DropPath regularisation
        • Returns patch embeddings optionally (for ensemble)
    """
    def __init__(
        self,
        d_model    = 512,
        nhead      = 8,
        num_layers = 6,
        dropout    = 0.1,
        drop_path  = 0.1,
        num_classes= 1,
        return_embeddings = False,
    ):
        super().__init__()
        self.return_embeddings = return_embeddings

        # ── CNN BACKBONE ─────────────────────────────────────────
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.cnn_backbone = nn.Sequential(*list(resnet.children())[:-2])
        for name, param in self.cnn_backbone.named_parameters():
            param.requires_grad = 'layer4' in name

        # ── PATCH PROJECTION ─────────────────────────────────────
        self.patch_proj    = nn.Linear(512, d_model)
        self.cls_token     = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_embedding = nn.Parameter(torch.randn(1, 50, d_model))
        self.drop_path     = DropPath(drop_path)

        # ── TRANSFORMER ───────────────────────────────────────────
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, activation='gelu',
            batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # ── CLASSIFIER HEAD ───────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
            nn.Sigmoid(),
        )

    def forward(self, x):
        B = x.shape[0]
        feat = self.cnn_backbone(x)                          # (B, 512, 7, 7)
        feat = feat.flatten(2).transpose(1, 2)              # (B, 49, 512)
        feat = self.patch_proj(feat)                         # (B, 49, d_model)
        cls  = self.cls_token.expand(B, -1, -1)             # (B, 1, d_model)
        feat = torch.cat([cls, feat], dim=1)                 # (B, 50, d_model)
        feat = feat + self.pos_embedding
        feat = self.transformer(feat)                        # (B, 50, d_model)
        cls_out = feat[:, 0, :]                             # (B, d_model)
        if self.return_embeddings:
            return self.classifier(cls_out), cls_out
        return self.classifier(cls_out)



In [5]:
#  CELL 4 — MODEL 2: CLIPWaveFormer (NEW)
#
#  WHAT IS CLIP?
#  ─────────────
#  CLIP (Contrastive Language–Image Pretraining, OpenAI 2021) was
#  trained on 400 million internet image-text pairs to align image
#  and text representations in a shared embedding space.
#
#  The image encoder is a Vision Transformer (ViT-B/32) that splits
#  an image into 7×7=49 non-overlapping patches of 32×32 pixels and
#  processes them with multi-head self-attention.
#
#  WHY USE CLIP FOR GRAVITATIONAL WAVES?
#  ─────────────────────────────────────
#  1. ViT patch embeddings are structurally different from ResNet
#     feature maps — they capture global relationships from layer 1.
#  2. CLIP's zero-shot capability lets us probe the model with text:
#     "a gravitational wave chirp" vs "random detector noise"
#     This is scientifically interesting and publishable.
#  3. CLIP embeddings generalise well to out-of-distribution signals
#     because they were trained on massive visual diversity.
#
#  ZERO-SHOT CLIP SCORING:
#  ────────────────────────
#  For each test spectrogram, compute cosine similarity between
#  its image embedding and two text embeddings:
#    • "a spectrogram showing a gravitational wave chirp signal"
#    • "a spectrogram showing random detector noise"
#  The softmax of similarities gives a zero-shot probability.
#  No fine-tuning needed — measure this BEFORE any training.

import open_clip

class CLIPWaveFormer(nn.Module):
    """
    CLIP ViT-B/32 image encoder + lightweight Transformer head.

    Architecture:
        Input (3, 224, 224)
            → CLIP ViT-B/32 encoder              → (B, 512) global embedding
                                                    OR patch tokens
            → Transformer head (2 layers)         → (B, 512)
            → Classifier                           → P(GW)

    We use open_clip (open-source reimplementation) which provides
    ViT-B/32-quickgelu pretrained on LAION-2B, a larger dataset than
    the original OpenAI CLIP training set.
    """
    def __init__(
        self,
        clip_model_name = 'ViT-B-32',
        pretrained      = 'laion2b_s34b_b79k',
        d_model         = 512,
        num_layers      = 2,
        nhead           = 8,
        dropout         = 0.1,
        num_classes     = 1,
        return_embeddings = False,
    ):
        super().__init__()
        self.return_embeddings = return_embeddings

        # ── CLIP IMAGE ENCODER ───────────────────────────────────
        # Load pretrained CLIP ViT-B/32
        # We extract ONLY the image encoder (visual transformer)
        # and freeze all but the last 4 blocks for fine-tuning.
        clip, _, preprocess = open_clip.create_model_and_transforms(
            clip_model_name, pretrained=pretrained)
        self.visual = clip.visual  # The ViT image encoder module

        # Freeze everything except the last 4 transformer blocks
        # CLIP ViT-B/32 has 12 transformer blocks (resblocks)
        for i, block in enumerate(self.visual.transformer.resblocks):
            freeze = (i < 8)    # Freeze blocks 0-7, fine-tune 8-11
            for param in block.parameters():
                param.requires_grad = not freeze

        # The CLIP ViT outputs a (B, 512) global CLS embedding
        # We also want intermediate patch tokens for the Transformer head.
        # Hook into the penultimate layer to capture patch tokens.
        self.patch_tokens = None
        def _hook(module, input, output):
            # output shape: (seq_len, B, dim) in CLIP's convention
            self.patch_tokens = output.permute(1, 0, 2)  # → (B, seq, dim)
        self.visual.transformer.resblocks[-1].register_forward_hook(_hook)

        # ── PROJECTION + TRANSFORMER HEAD ────────────────────────
        # CLIP ViT-B/32 outputs 512-dim embeddings; project if needed
        clip_dim = 768  # CLIP ViT-B/32 outputs 768-dim
        self.proj = nn.Linear(clip_dim, d_model) if clip_dim != d_model else nn.Identity()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 2,
            dropout=dropout, activation='gelu',
            batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # ── CLASSIFIER HEAD ───────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
            nn.Sigmoid(),
        )

    def forward(self, x):
        B = x.shape[0]

        # Run CLIP image encoder — this triggers the hook,
        # storing intermediate patch tokens in self.patch_tokens
        _ = self.visual(x)                                  # (B, 512)
        tokens = self.patch_tokens                           # (B, seq, 512)
        tokens = self.proj(tokens)                           # (B, seq, d_model)
        tokens = self.transformer(tokens)                    # (B, seq, d_model)
        cls_out = tokens[:, 0, :]                           # (B, d_model) — CLS token
        if self.return_embeddings:
            return self.classifier(cls_out), cls_out
        return self.classifier(cls_out)

    @torch.no_grad()
    def zero_shot_score(self, images, tokenizer=None):
        """
        Zero-shot gravitational wave detection using CLIP text prompts.

        Computes cosine similarity between each image embedding and two
        text embeddings, then returns a softmax probability for GW.

        This is a BASELINE — no fine-tuning. Run this BEFORE training
        to establish how much domain knowledge CLIP already has.

        Returns:
            gw_probs : (B,) tensor of P(GW) from zero-shot CLIP scoring
        """
        import open_clip
        tokenizer = open_clip.get_tokenizer('ViT-B-32')

        prompts = [
            "a time-frequency spectrogram showing a gravitational wave chirp signal rising in frequency",
            "a time-frequency spectrogram showing random detector noise with no gravitational wave",
        ]
        text_tokens = tokenizer(prompts).to(images.device)

        # Get image features
        img_feat  = self.visual(images)                     # (B, 512)
        img_feat  = F.normalize(img_feat, dim=-1)

        # Get text features from a fresh CLIP model
        # (reuse the visual encoder's parent CLIP model if available)
        # For simplicity, encode text via a temporary model instance
        clip_tmp, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
        clip_tmp       = clip_tmp.to(images.device).eval()
        text_feat      = clip_tmp.encode_text(text_tokens)  # (2, 512)
        text_feat      = F.normalize(text_feat, dim=-1)

        # Cosine similarity and softmax
        sim       = (img_feat @ text_feat.T) * 100          # (B, 2) scaled logits
        probs     = F.softmax(sim, dim=-1)                  # (B, 2)
        gw_probs  = probs[:, 0]                             # P(GW) for each image

        return gw_probs



In [6]:
#   CELL 5 — MODEL 3: WaveCNN1D
#
#  MOTIVATION: CAN WE DETECT GWs WITHOUT A SPECTROGRAM?
#  ──────────────────────────────────────────────────────
#  GravWaveFormer and CLIPWaveFormer both start from a Q-transform
#  spectrogram — a 2D representation that already involves a
#  design choice (window size, frequency resolution, etc.).
#
#  WaveCNN1D asks: what if we skip the spectrogram entirely and
#  learn directly from the bandpass-filtered 1D time-domain signal?
#  This model can detect patterns that may be invisible in the
#  time-frequency domain, and serves as a pure temporal baseline.
#
#  ARCHITECTURE: DILATED TEMPORAL CNN
#  ─────────────────────────────────────
#  Inspired by WaveNet and TCN (Temporal Convolutional Network):
#  • 1D convolutions with exponentially increasing dilation rates
#    (1, 2, 4, 8, 16, 32) to capture both fine-grained and long-
#    range temporal structure without losing resolution
#  • Residual connections prevent gradient vanishing in deep networks
#  • Three parallel "towers" (one per detector) with shared weights,
#    then cross-detector fusion via a 1D conv over the channel dim
#
#  WHY DILATION?
#  ─────────────
#  A standard conv with kernel_size=3 sees only 3 time steps.
#  A dilated conv with dilation=32 sees 65 time steps with only
#  3 weights — efficient for long-range temporal dependencies.
#  Stacking dilations 1,2,4,...,32 gives a receptive field of
#  2×(1+2+4+...+32) + 1 = 127 time steps per residual block.

class DilatedResBlock1D(nn.Module):
    """
    Single residual block with dilated causal 1D convolution.

    Structure (per block):
        Input → Conv1D(dilation) → BN → GELU → Conv1D(1) → BN
             ↓                                              ↑
             └──── skip connection (1×1 conv if dims differ)┘
    """
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        pad = dilation * (kernel_size - 1) // 2  # same-padding with dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size,
                               dilation=dilation, padding=pad, bias=False)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 1, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.act   = nn.GELU()
        self.drop  = nn.Dropout(dropout)
        self.skip  = nn.Conv1d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        h = self.act(self.bn1(self.conv1(x)))
        h = self.bn2(self.conv2(h))
        h = self.drop(h)
        return self.act(h + self.skip(x))


class WaveCNN1D(nn.Module):
    """
    Dilated temporal CNN operating on raw 1D LIGO strain data.

    Architecture:
        Input (3, 4096)                     — 3 detectors, 4096 time steps
            → Stem conv: 3 → 64 channels    — (64, 4096)
            → 6 DilatedResBlocks            — (256, 512) after pooling
              dilations: [1, 2, 4, 8, 16, 32]
              channels:  [64, 64, 128, 128, 256, 256]
            → Global average pooling        — (256,)
            → Classifier head               → P(GW)

    Total trainable parameters: ~1.5M
    Input: tensor of shape (B, 3, 4096)
    Output: tensor of shape (B, 1)
    """
    def __init__(self, num_classes=1, dropout=0.1, return_embeddings=False):
        super().__init__()
        self.return_embeddings = return_embeddings

        # ── STEM ──────────────────────────────────────────────────
        # First conv: expand from 3 channels (detectors) to 64
        # Uses a large kernel (15) to capture multi-detector structure
        self.stem = nn.Sequential(
            nn.Conv1d(3, 64, kernel_size=15, padding=7, bias=False),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.MaxPool1d(2),   # 4096 → 2048
        )

        # ── DILATED RESIDUAL TOWER ────────────────────────────────
        # Each block doubles the dilation to increase receptive field
        self.blocks = nn.Sequential(
            DilatedResBlock1D( 64,  64, dilation=1,  dropout=dropout),
            DilatedResBlock1D( 64,  64, dilation=2,  dropout=dropout),
            nn.MaxPool1d(2),   # 2048 → 1024
            DilatedResBlock1D( 64, 128, dilation=4,  dropout=dropout),
            DilatedResBlock1D(128, 128, dilation=8,  dropout=dropout),
            nn.MaxPool1d(2),   # 1024 → 512
            DilatedResBlock1D(128, 256, dilation=16, dropout=dropout),
            DilatedResBlock1D(256, 256, dilation=32, dropout=dropout),
            nn.MaxPool1d(2),   # 512 → 256
        )

        # ── TEMPORAL ATTENTION POOLING ────────────────────────────
        # Instead of naive global average pool, learn which time steps
        # are most informative for the classification decision.
        # This is equivalent to a 1-head temporal self-attention on the
        # feature sequence, collapsed to a single vector.
        self.temporal_attn = nn.Sequential(
            nn.Conv1d(256, 64, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(64, 1, kernel_size=1),    # (B, 1, T) attention logits
        )

        # ── CLASSIFIER HEAD ───────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.LayerNorm(256),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
            nn.Sigmoid(),
        )

    def forward(self, x):
        B = x.shape[0]
        h = self.stem(x)          # (B, 64,  2048)
        h = self.blocks(h)        # (B, 256, 256)

        # Temporal attention pooling
        attn_w = self.temporal_attn(h)   # (B, 1, 256)
        attn_w = F.softmax(attn_w, dim=-1)
        h_pool = (h * attn_w).sum(dim=-1)  # (B, 256)

        if self.return_embeddings:
            return self.classifier(h_pool), h_pool
        return self.classifier(h_pool)



In [7]:
#   CELL 6 — MODEL 4: CrossDetectorGNN
#
#  THE PHYSICS MOTIVATION
#  ─────────────────────
#  A real gravitational wave MUST arrive at all three LIGO/Virgo
#  detectors within 26 milliseconds (the light-travel time across
#  the detector triangle). A detector glitch — a local disturbance
#  like scattered light or seismic noise — appears in ONE detector
#  only.
#
#  Current approach (in GravWaveFormer): stack all 3 detector
#  channels as an RGB image. The model implicitly learns cross-
#  detector consistency but it is never explicitly modelled.
#
#  CrossDetectorGNN makes it EXPLICIT:
#  • Each detector is a NODE in a fully connected graph
#  • Edges encode the cross-correlation between detector pairs
#  • GNN message passing propagates signals between nodes
#  • After 2 rounds of message passing, the graph readout
#    captures whether signals are consistently coherent
#
#  THE GRAPH STRUCTURE
#  ───────────────────
#  Nodes:  {H1 (Hanford), L1 (Livingston), V1 (Virgo)} → 3 nodes
#  Edges:  fully connected → 3 × 2 = 6 directed edges
#  Node features: per-detector feature vector extracted from
#    a small 1D CNN (128-dim embedding per detector)
#  Edge features: normalised cross-correlation between each pair
#    (computed from raw waveform segments)
#
#  GNN ARCHITECTURE: GraphSAGE
#  ────────────────────────────
#  GraphSAGE (Hamilton et al., 2017) aggregates neighbour features
#  via mean pooling, then concatenates with the node's own features.
#  It is inductive (generalises to new graph structures) and simple
#  enough to train in minutes on 3-node graphs.
#
#  MESSAGE PASSING EQUATION:
#  h_v^(k) = W × CONCAT(h_v^(k-1), MEAN_{u ∈ N(v)} h_u^(k-1))
#  where v is a node, N(v) its neighbours, k the layer index.

try:
    from torch_geometric.nn import SAGEConv, global_mean_pool
    from torch_geometric.data import Data, Batch
    GEO_AVAILABLE = True
    print("✓ torch_geometric available — CrossDetectorGNN will use SAGEConv")
except ImportError:
    GEO_AVAILABLE = False
    print("⚠ torch_geometric not found — using manual GNN fallback")


class DetectorEncoder(nn.Module):
    """
    Encodes a single detector's 1D waveform into a 128-dim feature vector.
    Shared weights across all 3 detectors — physical symmetry assumption.
    """
    def __init__(self, in_len=4096, out_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, padding=7, bias=False),
            nn.BatchNorm1d(32), nn.GELU(),
            nn.MaxPool1d(4),          # 4096 → 1024
            nn.Conv1d(32, 64, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(64), nn.GELU(),
            nn.MaxPool1d(4),          # 1024 → 256
            nn.Conv1d(64, 128, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(128), nn.GELU(),
            nn.AdaptiveAvgPool1d(1),  # → 128
        )
        self.proj = nn.Linear(128, out_dim)

    def forward(self, x):
        # x: (B, time_len) — single detector waveform
        h = self.encoder(x.unsqueeze(1))  # (B, 128, 1)
        h = h.squeeze(-1)                 # (B, 128)
        return self.proj(h)               # (B, out_dim)


def compute_cross_correlation(x, max_lag=200):
    """
    Compute normalised cross-correlation between every pair of detectors.

    Parameters:
        x : (B, 3, T) — batch of 3-detector waveforms
        max_lag : int — max lag in samples to consider (±max_lag)

    Returns:
        corr_feats : (B, 3, 2*max_lag+1) cross-correlation per pair
        max_corr   : (B, 3) max correlation coefficient per pair

    Pairs: (H1-L1, H1-V1, L1-V1) → 3 pairs
    """
    B, C, T = x.shape
    pairs = [(0, 1), (0, 2), (1, 2)]
    max_corrs = []

    for (i, j) in pairs:
        # Normalise each channel
        xi = x[:, i, :]                             # (B, T)
        xj = x[:, j, :]
        xi = (xi - xi.mean(dim=1, keepdim=True)) / (xi.std(dim=1, keepdim=True) + 1e-8)
        xj = (xj - xj.mean(dim=1, keepdim=True)) / (xj.std(dim=1, keepdim=True) + 1e-8)

        # Cross-correlation via FFT (O(T log T) vs O(T²))
        Fx = torch.fft.rfft(xi, n=2*T)
        Fy = torch.fft.rfft(xj, n=2*T)
        cc = torch.fft.irfft(Fx * Fy.conj(), n=2*T)[:, :T] / T
        max_corr = cc.abs().max(dim=1).values          # (B,)
        max_corrs.append(max_corr)

    return torch.stack(max_corrs, dim=1)               # (B, 3)


class CrossDetectorGNN(nn.Module):
    """
    Graph Neural Network over the 3 LIGO/Virgo detectors.

    Pipeline:
        Input: (B, 3, T) raw waveforms — 3 detectors × T time steps
            → DetectorEncoder (shared weights)     → (B, 3, 128) node feats
            → compute_cross_correlation            → (B, 3) edge feats
            → 2× GraphSAGE message passing         → (B, 3, 128) updated nodes
            → Global mean pool + edge feat concat  → (B, 256)
            → Classifier head                      → P(GW)

    Edge features (cross-correlations) modulate message weights:
        higher correlation between two detectors → stronger message

    If torch_geometric is unavailable, falls back to a manual
    mean-pool GNN implementation that is functionally equivalent.
    """
    def __init__(self, node_dim=128, hidden_dim=128, num_classes=1,
                 dropout=0.1, return_embeddings=False):
        super().__init__()
        self.return_embeddings = return_embeddings

        # Shared encoder for all 3 detectors
        self.detector_enc = DetectorEncoder(out_dim=node_dim)

        # Edge feature projection (3 pair correlations → 32-dim edge embedding)
        self.edge_proj = nn.Linear(3, 32)

        if GEO_AVAILABLE:
            # GraphSAGE layers
            self.conv1 = SAGEConv(node_dim, hidden_dim)
            self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        else:
            # Fallback: manual mean-pool message passing
            self.agg1 = nn.Linear(node_dim * 2, hidden_dim)
            self.agg2 = nn.Linear(hidden_dim * 2, hidden_dim)

        # Classifier takes concatenated: graph readout + edge features
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim + 32),
            nn.Linear(hidden_dim + 32, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
            nn.Sigmoid(),
        )

        # Fully connected edge index for 3 nodes (used by torch_geometric)
        self.register_buffer(
            'edge_index',
            torch.tensor([[0,0,1,1,2,2],
                          [1,2,0,2,0,1]], dtype=torch.long)
        )

    def forward(self, x):
        """
        x: (B, 3, T) — batch of 3-detector waveforms
        """
        B = x.shape[0]

        # 1. Encode each detector independently
        node_feats = []
        for d in range(3):
            node_feats.append(self.detector_enc(x[:, d, :]))  # (B, node_dim)
        node_feats = torch.stack(node_feats, dim=1)            # (B, 3, node_dim)

        # 2. Compute edge features (cross-correlations)
        edge_feats = compute_cross_correlation(x)              # (B, 3)
        edge_emb   = F.gelu(self.edge_proj(edge_feats))       # (B, 32)

        if GEO_AVAILABLE:
            # 3. Build a batch of graphs for torch_geometric
            # Flatten (B, 3, node_dim) to (B*3, node_dim)
            node_flat = node_feats.view(B * 3, -1)            # (B*3, node_dim)

            # Offset edge index for each graph in the batch
            batch_vec   = torch.arange(B, device=x.device).repeat_interleave(3)
            edge_batched = torch.cat([
                self.edge_index + 3 * i for i in range(B)
            ], dim=1)                                          # (2, B*6)

            # Message passing
            h = F.gelu(self.conv1(node_flat, edge_batched))   # (B*3, hidden)
            h = F.gelu(self.conv2(h, edge_batched))           # (B*3, hidden)
            h = global_mean_pool(h, batch_vec)                 # (B, hidden)

        else:
            # Manual mean-pool GNN fallback
            # Layer 1: for each node, aggregate mean of neighbours
            h = node_feats                                     # (B, 3, node_dim)
            for _ in range(2):
                agg = h.mean(dim=1, keepdim=True).expand_as(h)  # mean of all nodes
                cat = torch.cat([h, agg], dim=-1)
                if h.shape[-1] == node_feats.shape[-1]:
                    h = F.gelu(self.agg1(cat))
                else:
                    h = F.gelu(self.agg2(cat))
            h = h.mean(dim=1)                                  # (B, hidden)

        # 4. Concatenate graph embedding + edge features → classify
        combined = torch.cat([h, edge_emb], dim=-1)           # (B, hidden+32)
        if self.return_embeddings:
            return self.classifier(combined), combined
        return self.classifier(combined)



⚠ torch_geometric not found — using manual GNN fallback


In [8]:
#  CELL 7 — MODEL 5: FlamingoBLIP2 VLM Explainer (NEW)
#
#  WHAT IS BLIP-2?
#  ───────────────
#  BLIP-2 (Bootstrapping Language-Image Pre-training, Li et al. 2023)
#  is a vision-language model that connects a frozen image encoder
#  (EVA-CLIP) and a frozen large language model (OPT or FlanT5)
#  via a lightweight Q-Former bridge module.
#
#  It is the most accessible approximation to Flamingo (DeepMind)
#  that can run on a Colab A100 using 8-bit quantization.
#
#  HOW WE USE IT:
#  ─────────────
#  BLIP-2 is a POST-HOC EXPLAINER, not a classifier.
#  After GravWaveEnsemble makes a binary prediction, we pass the
#  spectrogram image to BLIP-2 with a structured text prompt and
#  get a natural language explanation of what the model "sees".
#
#  EXAMPLE PROMPTS AND EXPECTED OUTPUTS:
#  ────────────────────────────────────────
#  Prompt:  "Describe the pattern in this LIGO spectrogram image.
#             Does it show a gravitational wave signal?"
#  GW:      "The image shows a sweeping curve rising from low to high
#             frequency, consistent with two massive objects spiralling
#             together and merging — a gravitational wave chirp."
#  Noise:   "The image shows irregular broadband noise with no coherent
#             frequency structure, consistent with detector background."
#
#  This output appears in the Streamlit app and Notebook 5 evaluation.
#
#  RUNTIME NOTE:
#  ─────────────
#  BLIP-2 requires ~8GB GPU RAM with 8-bit quantization.
#  Use Colab A100 (High-RAM) runtime for BLIP-2 inference.
#  T4 (free tier) can run BLIP-2 with load_in_8bit=True but slowly.

from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image
import torchvision.transforms as T

class FlamingoBLIP2Explainer:
    """
    BLIP-2 Vision-Language Model for spectrogram explanation.

    This is NOT a PyTorch nn.Module — it wraps a HuggingFace
    pipeline and is used ONLY for inference, not training.

    Usage:
        explainer = FlamingoBLIP2Explainer()
        explainer.load()
        text = explainer.explain(spectrogram_tensor, prediction_prob)
        print(text)
    """

    # Prompts for different prediction contexts
    PROMPTS = {
        'high_confidence_gw': (
            "Question: This is a LIGO gravitational wave detector spectrogram "
            "showing frequency (y-axis) vs time (x-axis). The detector found "
            "this signal highly likely to be a gravitational wave event. "
            "Describe the key visual features that indicate a gravitational wave chirp. "
            "Answer:"
        ),
        'high_confidence_noise': (
            "Question: This is a LIGO gravitational wave detector spectrogram "
            "showing frequency (y-axis) vs time (x-axis). The detector classified "
            "this as background noise with no gravitational wave present. "
            "Describe the visual characteristics that indicate random noise. "
            "Answer:"
        ),
        'uncertain': (
            "Question: This is a LIGO gravitational wave detector spectrogram "
            "showing frequency (y-axis) vs time (x-axis). The detector is uncertain "
            "whether this contains a gravitational wave. "
            "Describe what you see and any ambiguous features. "
            "Answer:"
        ),
        'generic': (
            "Question: Describe the most prominent visual pattern in this "
            "time-frequency spectrogram from the LIGO gravitational wave detector. "
            "Answer:"
        ),
    }

    def __init__(self, model_name='Salesforce/blip2-opt-2.7b'):
        """
        model_name options:
        • 'Salesforce/blip2-opt-2.7b'     ← 2.7B LLM, fits on T4 (8-bit)
        • 'Salesforce/blip2-opt-6.7b'     ← 6.7B LLM, needs A100
        • 'Salesforce/blip2-flan-t5-xl'   ← FlanT5 backbone, good at Q&A
        """
        self.model_name = model_name
        self.model      = None
        self.processor  = None
        self.loaded     = False

        # Inverse normalisation transform to convert spectrogram
        # tensor back to a displayable PIL image for BLIP-2
        self.to_pil = T.Compose([
            T.Normalize(
                mean=[-m/s for m, s in zip([0.485,0.456,0.406],
                                           [0.229,0.224,0.225])],
                std=[1/s for s in [0.229, 0.224, 0.225]]
            ),
            T.ToPILImage(),
        ])

    def load(self, load_in_8bit=True):
        """
        Load BLIP-2 model and processor from HuggingFace Hub.
        Call this ONCE before any inference — it loads ~5-12 GB.

        load_in_8bit=True: quantises to 8-bit, halving memory usage.
        Set False on A100 for faster inference with full precision.
        """
        print(f"Loading BLIP-2 ({self.model_name})...")
        print("  This downloads ~5-12 GB on first run (cached thereafter)")

        self.processor = Blip2Processor.from_pretrained(self.model_name)
        self.model = Blip2ForConditionalGeneration.from_pretrained(
            self.model_name,
            load_in_8bit = load_in_8bit,
            device_map   = 'auto',         # Automatically assigns layers to GPU/CPU
            torch_dtype  = torch.float16,
        )
        self.model.eval()
        self.loaded = True
        print(f"✓ BLIP-2 loaded ({self.model_name})")

    def _select_prompt(self, prob):
        """Choose a prompt based on the model's confidence."""
        if prob > 0.80:
            return self.PROMPTS['high_confidence_gw']
        elif prob < 0.20:
            return self.PROMPTS['high_confidence_noise']
        elif 0.35 < prob < 0.65:
            return self.PROMPTS['uncertain']
        else:
            return self.PROMPTS['generic']

    @torch.no_grad()
    def explain(self, spectrogram_tensor, prediction_prob, max_new_tokens=120):
        """
        Generate a natural language explanation for one spectrogram.

        Parameters:
            spectrogram_tensor : (3, 224, 224) float tensor
            prediction_prob    : float — P(GW) from the ensemble model
            max_new_tokens     : int — max tokens to generate

        Returns:
            explanation : str — natural language description
        """
        if not self.loaded:
            raise RuntimeError("Call .load() before .explain()")

        # Convert tensor to PIL Image
        img    = self.to_pil(spectrogram_tensor.cpu().clamp(0, 1))
        prompt = self._select_prompt(prediction_prob)

        # Tokenise and run inference
        inputs = self.processor(
            images  = img,
            text    = prompt,
            return_tensors = 'pt',
        ).to(next(self.model.parameters()).device)

        output_ids  = self.model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            num_beams      = 4,
            temperature    = 0.7,
            repetition_penalty = 1.2,
        )
        explanation = self.processor.decode(
            output_ids[0], skip_special_tokens=True
        ).strip()

        # Strip the prompt prefix if it was echoed back
        if prompt.rstrip() in explanation:
            explanation = explanation.replace(prompt.rstrip(), '').strip()

        return explanation

    @torch.no_grad()
    def explain_batch(self, spectrogram_tensors, prediction_probs, max_new_tokens=120):
        """
        Generate explanations for a batch of spectrograms.

        Parameters:
            spectrogram_tensors : list of (3, 224, 224) tensors
            prediction_probs    : list of float probabilities

        Returns:
            explanations : list of str
        """
        if not self.loaded:
            raise RuntimeError("Call .load() before .explain_batch()")

        explanations = []
        for i, (tensor, prob) in enumerate(zip(spectrogram_tensors, prediction_probs)):
            print(f"  Explaining sample {i+1}/{len(spectrogram_tensors)}...", end='\r')
            explanation = self.explain(tensor, float(prob), max_new_tokens)
            explanations.append(explanation)
        print()
        return explanations



In [9]:
#   CELL 8 — ENSEMBLE: GravWaveEnsemble
#
#  ENSEMBLE STRATEGY: MLP META-LEARNER (STACKING)
#  ───────────────────────────────────────────────
#  Each individual model produces a probability P(GW) ∈ [0, 1].
#  The ensemble combines them in two ways:
#
#  Method A — Simple averaging (no learnable params):
#    P_ensemble = mean(P_gravwave, P_clip, P_wave1d, P_gnn)
#    Fast, robust, no risk of overfitting the ensemble.
#    Good for reporting a quick combined metric.
#
#  Method B — MLP meta-learner (stacking):
#    The four probabilities AND their intermediate embeddings
#    are concatenated and fed to a 2-layer MLP trained to
#    optimally weight each model's contribution.
#    Typically +1–3% AUC over simple averaging.
#
#  TRAINING STRATEGY:
#  ──────────────────
#  1. Train each component model independently (Notebook 4)
#  2. Generate embedding predictions on a validation hold-out set
#  3. Train the MLP meta-learner on those embeddings (20 epochs)
#  4. The full forward pass (all 4 models + MLP) is the final system
#
#  EMBEDDING DIMENSIONS USED:
#    GravWaveFormer CLS :  512
#    CLIPWaveFormer CLS :  512
#    WaveCNN1D pool     :  256
#    CrossDetectorGNN   :  128 + 32 = 160
#    ─────────────────────────
#    Total              : 1440

class GravWaveEnsemble(nn.Module):
    """
    Multi-model ensemble for gravitational wave detection.

    Combines GravWaveFormer, CLIPWaveFormer, WaveCNN1D, and
    CrossDetectorGNN via a trained MLP meta-learner.

    BLIP-2 is handled separately (inference only, not part of
    the differentiable computation graph).

    Forward pass requires:
        spec_2d   : (B, 3, 224, 224) — spectrogram images
        wave_1d   : (B, 3, 4096)     — raw 1D waveforms

    Both representations come from the same LIGO window.
    See Notebook 2 (preprocessing) for how they are saved.
    """
    def __init__(
        self,
        gravwave_ckpt = None,
        clip_ckpt     = None,
        wave1d_ckpt   = None,
        gnn_ckpt      = None,
        ensemble_mode = 'mlp',      # 'mlp' or 'average'
    ):
        super().__init__()
        self.ensemble_mode = ensemble_mode

        # ── COMPONENT MODELS (with embedding return enabled) ──────
        self.gravwave = GravWaveFormer(return_embeddings=True)
        self.clip     = CLIPWaveFormer(return_embeddings=True)
        self.wave1d   = WaveCNN1D(return_embeddings=True)
        self.gnn      = CrossDetectorGNN(return_embeddings=True)

        # Load pretrained weights if provided
        if gravwave_ckpt:
            self.gravwave.load_state_dict(
                torch.load(gravwave_ckpt, map_location='cpu')['model_state_dict'])
        if clip_ckpt:
            self.clip.load_state_dict(
                torch.load(clip_ckpt, map_location='cpu')['model_state_dict'])
        if wave1d_ckpt:
            self.wave1d.load_state_dict(
                torch.load(wave1d_ckpt, map_location='cpu')['model_state_dict'])
        if gnn_ckpt:
            self.gnn.load_state_dict(
                torch.load(gnn_ckpt, map_location='cpu')['model_state_dict'])

        # ── MLP META-LEARNER ──────────────────────────────────────
        # Input: 4 raw probabilities (4) + concatenated embeddings (1440)
        # Total input size: 4 + 512 + 512 + 256 + 160 = 1444
        EMB_DIM = 4 + 512 + 512 + 256 + 160

        self.meta_learner = nn.Sequential(
            nn.LayerNorm(EMB_DIM),
            nn.Linear(EMB_DIM, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

        # ── ATTENTION WEIGHTS (for interpretability) ──────────────
        # Learnable weights that assign an importance score to each
        # component model. Softmax-normalised → sum to 1.
        # Used in ensemble_mode='weighted_average'.
        self.model_weights = nn.Parameter(torch.ones(4))

    def forward(self, spec_2d, wave_1d):
        """
        Full ensemble forward pass.

        Parameters:
            spec_2d : (B, 3, 224, 224)
            wave_1d : (B, 3, 4096)

        Returns:
            prob    : (B, 1) final ensemble probability
            details : dict with per-model probabilities (for analysis)
        """
        # ── INDIVIDUAL MODEL PREDICTIONS ─────────────────────────
        p_gw,  e_gw  = self.gravwave(spec_2d)    # (B,1), (B,512)
        p_clip, e_clip = self.clip(spec_2d)       # (B,1), (B,512)
        p_1d,  e_1d  = self.wave1d(wave_1d)      # (B,1), (B,256)
        p_gnn, e_gnn = self.gnn(wave_1d)          # (B,1), (B,160)

        details = {
            'p_gravwave': p_gw.squeeze(1),
            'p_clip':     p_clip.squeeze(1),
            'p_wave1d':   p_1d.squeeze(1),
            'p_gnn':      p_gnn.squeeze(1),
        }

        if self.ensemble_mode == 'average':
            # Simple unweighted average
            prob = (p_gw + p_clip + p_1d + p_gnn) / 4
            return prob, details

        elif self.ensemble_mode == 'weighted_average':
            # Softmax-weighted average (weights are learnable)
            w    = F.softmax(self.model_weights, dim=0)
            probs = torch.cat([p_gw, p_clip, p_1d, p_gnn], dim=1)   # (B, 4)
            prob  = (probs * w).sum(dim=1, keepdim=True)
            details['model_weights'] = w.detach()
            return prob, details

        else:
            # 'mlp' — MLP meta-learner on probs + embeddings
            raw_probs = torch.cat([p_gw, p_clip, p_1d, p_gnn], dim=1)  # (B, 4)
            embeddings = torch.cat([e_gw, e_clip, e_1d, e_gnn], dim=1) # (B, 1440)
            combined   = torch.cat([raw_probs, embeddings], dim=1)      # (B, 1444)
            prob       = self.meta_learner(combined)                     # (B, 1)
            return prob, details

    def freeze_backbones(self):
        """
        Freeze all component model weights.
        Call this when training ONLY the meta-learner.
        """
        for m in [self.gravwave, self.clip, self.wave1d, self.gnn]:
            for p in m.parameters():
                p.requires_grad = False
        print("✓ All backbone models frozen. Only meta-learner is trainable.")

    def unfreeze_backbones(self):
        """
        Unfreeze all component model weights.
        Call this for final end-to-end fine-tuning of the full system.
        """
        for m in [self.gravwave, self.clip, self.wave1d, self.gnn]:
            for p in m.parameters():
                p.requires_grad = True
        print("✓ All backbone models unfrozen. Full system is trainable.")

    def get_model_importances(self):
        """
        Returns a dict showing each model's current learned importance
        (only meaningful for ensemble_mode='weighted_average').
        """
        w = F.softmax(self.model_weights, dim=0).detach().cpu()
        return {
            'GravWaveFormer': float(w[0]),
            'CLIPWaveFormer': float(w[1]),
            'WaveCNN1D':      float(w[2]),
            'CrossDetectorGNN': float(w[3]),
        }



In [10]:
#   CELL 9 — ARCHITECTURE TESTS AND PARAMETER COUNTS

def count_parameters(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def run_architecture_tests():
    """
    Instantiates each model and runs a test forward pass.
    Reports parameter counts and output shapes.
    """
    print("\n" + "=" * 65)
    print("  ARCHITECTURE TESTS — ALL MODELS")
    print("=" * 65)

    dummy_2d = torch.randn(4, 3, 224, 224).to(device)  # batch of 4 spectrograms
    dummy_1d = torch.randn(4, 3, 4096).to(device)       # batch of 4 waveforms

    results = []

    # ── MODEL 1: GravWaveFormer ───────────────────────────────────
    print("\n[1/4] GravWaveFormer")
    m1 = GravWaveFormer(return_embeddings=True).to(device)
    out, emb = m1(dummy_2d)
    total, trainable = count_parameters(m1)
    print(f"  Input shape  : {tuple(dummy_2d.shape)}")
    print(f"  Output shape : {tuple(out.shape)}  (prob)  {tuple(emb.shape)}  (embedding)")
    print(f"  Total params : {total:,}  |  Trainable: {trainable:,}")
    results.append(('GravWaveFormer', total, trainable))
    del m1

    # ── MODEL 2: CLIPWaveFormer ───────────────────────────────────
    print("\n[2/4] CLIPWaveFormer")
    try:
        m2 = CLIPWaveFormer(return_embeddings=True).to(device)
        out, emb = m2(dummy_2d)
        total, trainable = count_parameters(m2)
        print(f"  Input shape  : {tuple(dummy_2d.shape)}")
        print(f"  Output shape : {tuple(out.shape)}  (prob)  {tuple(emb.shape)}  (embedding)")
        print(f"  Total params : {total:,}  |  Trainable: {trainable:,}")
        results.append(('CLIPWaveFormer', total, trainable))
        del m2
    except Exception as e:
        print(f"  ⚠ CLIP test failed: {e}")
        print("  Ensure open-clip-torch is installed: pip install open-clip-torch")

    # ── MODEL 3: WaveCNN1D ────────────────────────────────────────
    print("\n[3/4] WaveCNN1D")
    m3 = WaveCNN1D(return_embeddings=True).to(device)
    out, emb = m3(dummy_1d)
    total, trainable = count_parameters(m3)
    print(f"  Input shape  : {tuple(dummy_1d.shape)}")
    print(f"  Output shape : {tuple(out.shape)}  (prob)  {tuple(emb.shape)}  (embedding)")
    print(f"  Total params : {total:,}  |  Trainable: {trainable:,}")
    results.append(('WaveCNN1D', total, trainable))
    del m3

    # ── MODEL 4: CrossDetectorGNN ─────────────────────────────────
    print("\n[4/4] CrossDetectorGNN")
    m4 = CrossDetectorGNN(return_embeddings=True).to(device)
    out, emb = m4(dummy_1d)
    total, trainable = count_parameters(m4)
    print(f"  Input shape  : {tuple(dummy_1d.shape)}")
    print(f"  Output shape : {tuple(out.shape)}  (prob)  {tuple(emb.shape)}  (embedding)")
    print(f"  Total params : {total:,}  |  Trainable: {trainable:,}")
    results.append(('CrossDetectorGNN', total, trainable))
    del m4

    # ── SUMMARY TABLE ─────────────────────────────────────────────
    print("\n" + "=" * 65)
    print("  PARAMETER SUMMARY")
    print("=" * 65)
    print(f"  {'Model':<22} {'Total Params':>15}  {'Trainable':>12}")
    print("  " + "-" * 50)
    total_all = 0
    train_all = 0
    for name, tot, tr in results:
        print(f"  {name:<22} {tot:>15,}  {tr:>12,}")
        total_all += tot
        train_all += tr
    print("  " + "-" * 50)
    print(f"  {'TOTAL (excl. ensemble)':<22} {total_all:>15,}  {train_all:>12,}")
    print()
    print("  NOTE: BLIP-2 (~2.7B params) is not counted above.")
    print("        It is used for inference only, never trained.")
    print()
    print("✓ All architecture tests passed!")
    return results

results = run_architecture_tests()




  ARCHITECTURE TESTS — ALL MODELS

[1/4] GravWaveFormer
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 213MB/s]
/tmp/ipykernel_11188/2668349737.py:79: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


  Input shape  : (4, 3, 224, 224)
  Output shape : (4, 1)  (prob)  (4, 512)  (embedding)
  Total params : 30,446,401  |  Trainable: 19,269,889

[2/4] CLIPWaveFormer


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/tmp/ipykernel_11188/3649078227.py:96: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


  Input shape  : (4, 3, 224, 224)
  Output shape : (50, 1)  (prob)  (50, 512)  (embedding)
  Total params : 92,515,329  |  Trainable: 35,812,353

[3/4] WaveCNN1D
  Input shape  : (4, 3, 4096)
  Output shape : (4, 1)  (prob)  (4, 256)  (embedding)
  Total params : 646,338  |  Trainable: 646,338

[4/4] CrossDetectorGNN
  Input shape  : (4, 3, 4096)
  Output shape : (4, 1)  (prob)  (4, 160)  (embedding)
  Total params : 149,345  |  Trainable: 149,345

  PARAMETER SUMMARY
  Model                     Total Params     Trainable
  --------------------------------------------------
  GravWaveFormer              30,446,401    19,269,889
  CLIPWaveFormer              92,515,329    35,812,353
  WaveCNN1D                      646,338       646,338
  CrossDetectorGNN               149,345       149,345
  --------------------------------------------------
  TOTAL (excl. ensemble)     123,757,413    55,877,925

  NOTE: BLIP-2 (~2.7B params) is not counted above.
        It is used for inference only,

In [11]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — SAVE MODEL CLASSES TO gravwave_models.py         ║
# ╚══════════════════════════════════════════════════════════════╝
#
# Writes gravwave_models.py directly — no inspect.getsource() needed.
# This works reliably in Colab regardless of session state.

save_path = f'{PROJECT_DIR}/gravwave_models.py'

# Full model code embedded directly as a string
file_content = '# GravWave-Former Model Definitions\n# Auto-generated by Notebook_03_Model.ipynb\n\nimport os, sys\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torchvision import models\nimport numpy as np\n\ntry:\n    import open_clip\n    OPEN_CLIP_AVAILABLE = True\nexcept ImportError:\n    OPEN_CLIP_AVAILABLE = False\n\ntry:\n    from torch_geometric.nn import SAGEConv, global_mean_pool\n    from torch_geometric.data import Data, Batch\n    GEO_AVAILABLE = True\nexcept ImportError:\n    GEO_AVAILABLE = False\n\ntry:\n    from transformers import Blip2Processor, Blip2ForConditionalGeneration\n    from PIL import Image\n    import torchvision.transforms as T\n    BLIP_AVAILABLE = True\nexcept ImportError:\n    BLIP_AVAILABLE = False\n\n#   CELL 3 — MODEL 1: GravWaveFormer (upgraded from v1)\n#\n#  Changes from v1:\n#  • d_model bumped from 256 → 512 for richer token representations\n#  • Transformer depth increased from 4 → 6 layers\n#  • Added stochastic depth (drop path) for stronger regularisation\n#  • forward() now optionally returns intermediate patch embeddings\n#    so the ensemble can access richer features than just CLS\n\nclass DropPath(nn.Module):\n    """\n    Stochastic Depth / Drop Path regularisation.\n    Randomly drops entire residual paths during training.\n    Stronger regulariser than standard dropout for Transformers.\n    Introduced in "Deep Networks with Stochastic Depth" (Huang et al.)\n    """\n    def __init__(self, drop_prob=0.0):\n        super().__init__()\n        self.drop_prob = drop_prob\n\n    def forward(self, x):\n        if not self.training or self.drop_prob == 0.0:\n            return x\n        keep_prob = 1 - self.drop_prob\n        shape      = (x.shape[0],) + (1,) * (x.ndim - 1)\n        rand_tensor = torch.rand(shape, dtype=x.dtype, device=x.device)\n        rand_tensor = torch.floor(rand_tensor + keep_prob)\n        return x / keep_prob * rand_tensor\n\n\nclass GravWaveFormer(nn.Module):\n    """\n    GravWave-Former: ResNet-18 CNN + deeper Transformer encoder.\n\n    Architecture:\n        Input (3, 224, 224)\n            → ResNet-18 (layer4 fine-tuned)      → (B, 512, 7, 7)\n            → Patch projection (512→d_model)      → (B, 49, d_model)\n            → Prepend [CLS] + positional encoding → (B, 50, d_model)\n            → 6-layer Transformer encoder\n            → CLS output → classifier → P(GW)\n\n    Changes from v1:\n        • d_model: 256 → 512\n        • num_layers: 4 → 6\n        • Added DropPath regularisation\n        • Returns patch embeddings optionally (for ensemble)\n    """\n    def __init__(\n        self,\n        d_model    = 512,\n        nhead      = 8,\n        num_layers = 6,\n        dropout    = 0.1,\n        drop_path  = 0.1,\n        num_classes= 1,\n        return_embeddings = False,\n    ):\n        super().__init__()\n        self.return_embeddings = return_embeddings\n\n        # ── CNN BACKBONE ─────────────────────────────────────────\n        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)\n        self.cnn_backbone = nn.Sequential(*list(resnet.children())[:-2])\n        for name, param in self.cnn_backbone.named_parameters():\n            param.requires_grad = \'layer4\' in name\n\n        # ── PATCH PROJECTION ─────────────────────────────────────\n        self.patch_proj    = nn.Linear(512, d_model)\n        self.cls_token     = nn.Parameter(torch.randn(1, 1, d_model))\n        self.pos_embedding = nn.Parameter(torch.randn(1, 50, d_model))\n        self.drop_path     = DropPath(drop_path)\n\n        # ── TRANSFORMER ───────────────────────────────────────────\n        encoder_layer = nn.TransformerEncoderLayer(\n            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,\n            dropout=dropout, activation=\'gelu\',\n            batch_first=True, norm_first=True)\n        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)\n\n        # ── CLASSIFIER HEAD ───────────────────────────────────────\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(d_model),\n            nn.Linear(d_model, 128),\n            nn.GELU(),\n            nn.Dropout(0.3),\n            nn.Linear(128, num_classes),\n            nn.Sigmoid(),\n        )\n\n    def forward(self, x):\n        B = x.shape[0]\n        feat = self.cnn_backbone(x)                          # (B, 512, 7, 7)\n        feat = feat.flatten(2).transpose(1, 2)              # (B, 49, 512)\n        feat = self.patch_proj(feat)                         # (B, 49, d_model)\n        cls  = self.cls_token.expand(B, -1, -1)             # (B, 1, d_model)\n        feat = torch.cat([cls, feat], dim=1)                 # (B, 50, d_model)\n        feat = feat + self.pos_embedding\n        feat = self.transformer(feat)                        # (B, 50, d_model)\n        cls_out = feat[:, 0, :]                             # (B, d_model)\n        if self.return_embeddings:\n            return self.classifier(cls_out), cls_out\n        return self.classifier(cls_out)\n\n\n\n#  CELL 4 — MODEL 2: CLIPWaveFormer (NEW)\n#\n#  WHAT IS CLIP?\n#  ─────────────\n#  CLIP (Contrastive Language–Image Pretraining, OpenAI 2021) was\n#  trained on 400 million internet image-text pairs to align image\n#  and text representations in a shared embedding space.\n#\n#  The image encoder is a Vision Transformer (ViT-B/32) that splits\n#  an image into 7×7=49 non-overlapping patches of 32×32 pixels and\n#  processes them with multi-head self-attention.\n#\n#  WHY USE CLIP FOR GRAVITATIONAL WAVES?\n#  ─────────────────────────────────────\n#  1. ViT patch embeddings are structurally different from ResNet\n#     feature maps — they capture global relationships from layer 1.\n#  2. CLIP\'s zero-shot capability lets us probe the model with text:\n#     "a gravitational wave chirp" vs "random detector noise"\n#     This is scientifically interesting and publishable.\n#  3. CLIP embeddings generalise well to out-of-distribution signals\n#     because they were trained on massive visual diversity.\n#\n#  ZERO-SHOT CLIP SCORING:\n#  ────────────────────────\n#  For each test spectrogram, compute cosine similarity between\n#  its image embedding and two text embeddings:\n#    • "a spectrogram showing a gravitational wave chirp signal"\n#    • "a spectrogram showing random detector noise"\n#  The softmax of similarities gives a zero-shot probability.\n#  No fine-tuning needed — measure this BEFORE any training.\n\nimport open_clip\n\nclass CLIPWaveFormer(nn.Module):\n    """\n    CLIP ViT-B/32 image encoder + lightweight Transformer head.\n\n    Architecture:\n        Input (3, 224, 224)\n            → CLIP ViT-B/32 encoder              → (B, 512) global embedding\n                                                    OR patch tokens\n            → Transformer head (2 layers)         → (B, 512)\n            → Classifier                           → P(GW)\n\n    We use open_clip (open-source reimplementation) which provides\n    ViT-B/32-quickgelu pretrained on LAION-2B, a larger dataset than\n    the original OpenAI CLIP training set.\n    """\n    def __init__(\n        self,\n        clip_model_name = \'ViT-B-32\',\n        pretrained      = \'laion2b_s34b_b79k\',\n        d_model         = 512,\n        num_layers      = 2,\n        nhead           = 8,\n        dropout         = 0.1,\n        num_classes     = 1,\n        return_embeddings = False,\n    ):\n        super().__init__()\n        self.return_embeddings = return_embeddings\n\n        # ── CLIP IMAGE ENCODER ───────────────────────────────────\n        # Load pretrained CLIP ViT-B/32\n        # We extract ONLY the image encoder (visual transformer)\n        # and freeze all but the last 4 blocks for fine-tuning.\n        clip, _, preprocess = open_clip.create_model_and_transforms(\n            clip_model_name, pretrained=pretrained)\n        self.visual = clip.visual  # The ViT image encoder module\n\n        # Freeze everything except the last 4 transformer blocks\n        # CLIP ViT-B/32 has 12 transformer blocks (resblocks)\n        for i, block in enumerate(self.visual.transformer.resblocks):\n            freeze = (i < 8)    # Freeze blocks 0-7, fine-tune 8-11\n            for param in block.parameters():\n                param.requires_grad = not freeze\n\n        # The CLIP ViT outputs a (B, 512) global CLS embedding\n        # We also want intermediate patch tokens for the Transformer head.\n        # Hook into the penultimate layer to capture patch tokens.\n        self.patch_tokens = None\n        def _hook(module, input, output):\n            # output shape: (seq_len, B, dim) in CLIP\'s convention\n            self.patch_tokens = output.permute(1, 0, 2)  # → (B, seq, dim)\n        self.visual.transformer.resblocks[-1].register_forward_hook(_hook)\n\n        # ── PROJECTION + TRANSFORMER HEAD ────────────────────────\n        # CLIP ViT-B/32 outputs 512-dim embeddings; project if needed\n        clip_dim = 768  # CLIP ViT-B/32 outputs 768-dim\n        self.proj = nn.Linear(clip_dim, d_model) if clip_dim != d_model else nn.Identity()\n\n        encoder_layer = nn.TransformerEncoderLayer(\n            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 2,\n            dropout=dropout, activation=\'gelu\',\n            batch_first=True, norm_first=True)\n        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)\n\n        # ── CLASSIFIER HEAD ───────────────────────────────────────\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(d_model),\n            nn.Linear(d_model, 128),\n            nn.GELU(),\n            nn.Dropout(0.3),\n            nn.Linear(128, num_classes),\n            nn.Sigmoid(),\n        )\n\n    def forward(self, x):\n        B = x.shape[0]\n\n        # Run CLIP image encoder — this triggers the hook,\n        # storing intermediate patch tokens in self.patch_tokens\n        _ = self.visual(x)                                  # (B, 512)\n        tokens = self.patch_tokens                           # (B, seq, 512)\n        tokens = self.proj(tokens)                           # (B, seq, d_model)\n        tokens = self.transformer(tokens)                    # (B, seq, d_model)\n        cls_out = tokens[:, 0, :]                           # (B, d_model) — CLS token\n        if self.return_embeddings:\n            return self.classifier(cls_out), cls_out\n        return self.classifier(cls_out)\n\n    @torch.no_grad()\n    def zero_shot_score(self, images, tokenizer=None):\n        """\n        Zero-shot gravitational wave detection using CLIP text prompts.\n\n        Computes cosine similarity between each image embedding and two\n        text embeddings, then returns a softmax probability for GW.\n\n        This is a BASELINE — no fine-tuning. Run this BEFORE training\n        to establish how much domain knowledge CLIP already has.\n\n        Returns:\n            gw_probs : (B,) tensor of P(GW) from zero-shot CLIP scoring\n        """\n        import open_clip\n        tokenizer = open_clip.get_tokenizer(\'ViT-B-32\')\n\n        prompts = [\n            "a time-frequency spectrogram showing a gravitational wave chirp signal rising in frequency",\n            "a time-frequency spectrogram showing random detector noise with no gravitational wave",\n        ]\n        text_tokens = tokenizer(prompts).to(images.device)\n\n        # Get image features\n        img_feat  = self.visual(images)                     # (B, 512)\n        img_feat  = F.normalize(img_feat, dim=-1)\n\n        # Get text features from a fresh CLIP model\n        # (reuse the visual encoder\'s parent CLIP model if available)\n        # For simplicity, encode text via a temporary model instance\n        clip_tmp, _, _ = open_clip.create_model_and_transforms(\'ViT-B-32\', pretrained=\'laion2b_s34b_b79k\')\n        clip_tmp       = clip_tmp.to(images.device).eval()\n        text_feat      = clip_tmp.encode_text(text_tokens)  # (2, 512)\n        text_feat      = F.normalize(text_feat, dim=-1)\n\n        # Cosine similarity and softmax\n        sim       = (img_feat @ text_feat.T) * 100          # (B, 2) scaled logits\n        probs     = F.softmax(sim, dim=-1)                  # (B, 2)\n        gw_probs  = probs[:, 0]                             # P(GW) for each image\n\n        return gw_probs\n\n\n\n#   CELL 5 — MODEL 3: WaveCNN1D\n#\n#  MOTIVATION: CAN WE DETECT GWs WITHOUT A SPECTROGRAM?\n#  ──────────────────────────────────────────────────────\n#  GravWaveFormer and CLIPWaveFormer both start from a Q-transform\n#  spectrogram — a 2D representation that already involves a\n#  design choice (window size, frequency resolution, etc.).\n#\n#  WaveCNN1D asks: what if we skip the spectrogram entirely and\n#  learn directly from the bandpass-filtered 1D time-domain signal?\n#  This model can detect patterns that may be invisible in the\n#  time-frequency domain, and serves as a pure temporal baseline.\n#\n#  ARCHITECTURE: DILATED TEMPORAL CNN\n#  ─────────────────────────────────────\n#  Inspired by WaveNet and TCN (Temporal Convolutional Network):\n#  • 1D convolutions with exponentially increasing dilation rates\n#    (1, 2, 4, 8, 16, 32) to capture both fine-grained and long-\n#    range temporal structure without losing resolution\n#  • Residual connections prevent gradient vanishing in deep networks\n#  • Three parallel "towers" (one per detector) with shared weights,\n#    then cross-detector fusion via a 1D conv over the channel dim\n#\n#  WHY DILATION?\n#  ─────────────\n#  A standard conv with kernel_size=3 sees only 3 time steps.\n#  A dilated conv with dilation=32 sees 65 time steps with only\n#  3 weights — efficient for long-range temporal dependencies.\n#  Stacking dilations 1,2,4,...,32 gives a receptive field of\n#  2×(1+2+4+...+32) + 1 = 127 time steps per residual block.\n\nclass DilatedResBlock1D(nn.Module):\n    """\n    Single residual block with dilated causal 1D convolution.\n\n    Structure (per block):\n        Input → Conv1D(dilation) → BN → GELU → Conv1D(1) → BN\n             ↓                                              ↑\n             └──── skip connection (1×1 conv if dims differ)┘\n    """\n    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.1):\n        super().__init__()\n        pad = dilation * (kernel_size - 1) // 2  # same-padding with dilation\n        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size,\n                               dilation=dilation, padding=pad, bias=False)\n        self.conv2 = nn.Conv1d(out_ch, out_ch, 1, bias=False)\n        self.bn1   = nn.BatchNorm1d(out_ch)\n        self.bn2   = nn.BatchNorm1d(out_ch)\n        self.act   = nn.GELU()\n        self.drop  = nn.Dropout(dropout)\n        self.skip  = nn.Conv1d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()\n\n    def forward(self, x):\n        h = self.act(self.bn1(self.conv1(x)))\n        h = self.bn2(self.conv2(h))\n        h = self.drop(h)\n        return self.act(h + self.skip(x))\n\n\nclass WaveCNN1D(nn.Module):\n    """\n    Dilated temporal CNN operating on raw 1D LIGO strain data.\n\n    Architecture:\n        Input (3, 4096)                     — 3 detectors, 4096 time steps\n            → Stem conv: 3 → 64 channels    — (64, 4096)\n            → 6 DilatedResBlocks            — (256, 512) after pooling\n              dilations: [1, 2, 4, 8, 16, 32]\n              channels:  [64, 64, 128, 128, 256, 256]\n            → Global average pooling        — (256,)\n            → Classifier head               → P(GW)\n\n    Total trainable parameters: ~1.5M\n    Input: tensor of shape (B, 3, 4096)\n    Output: tensor of shape (B, 1)\n    """\n    def __init__(self, num_classes=1, dropout=0.1, return_embeddings=False):\n        super().__init__()\n        self.return_embeddings = return_embeddings\n\n        # ── STEM ──────────────────────────────────────────────────\n        # First conv: expand from 3 channels (detectors) to 64\n        # Uses a large kernel (15) to capture multi-detector structure\n        self.stem = nn.Sequential(\n            nn.Conv1d(3, 64, kernel_size=15, padding=7, bias=False),\n            nn.BatchNorm1d(64),\n            nn.GELU(),\n            nn.MaxPool1d(2),   # 4096 → 2048\n        )\n\n        # ── DILATED RESIDUAL TOWER ────────────────────────────────\n        # Each block doubles the dilation to increase receptive field\n        self.blocks = nn.Sequential(\n            DilatedResBlock1D( 64,  64, dilation=1,  dropout=dropout),\n            DilatedResBlock1D( 64,  64, dilation=2,  dropout=dropout),\n            nn.MaxPool1d(2),   # 2048 → 1024\n            DilatedResBlock1D( 64, 128, dilation=4,  dropout=dropout),\n            DilatedResBlock1D(128, 128, dilation=8,  dropout=dropout),\n            nn.MaxPool1d(2),   # 1024 → 512\n            DilatedResBlock1D(128, 256, dilation=16, dropout=dropout),\n            DilatedResBlock1D(256, 256, dilation=32, dropout=dropout),\n            nn.MaxPool1d(2),   # 512 → 256\n        )\n\n        # ── TEMPORAL ATTENTION POOLING ────────────────────────────\n        # Instead of naive global average pool, learn which time steps\n        # are most informative for the classification decision.\n        # This is equivalent to a 1-head temporal self-attention on the\n        # feature sequence, collapsed to a single vector.\n        self.temporal_attn = nn.Sequential(\n            nn.Conv1d(256, 64, kernel_size=1),\n            nn.GELU(),\n            nn.Conv1d(64, 1, kernel_size=1),    # (B, 1, T) attention logits\n        )\n\n        # ── CLASSIFIER HEAD ───────────────────────────────────────\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(256),\n            nn.Linear(256, 64),\n            nn.GELU(),\n            nn.Dropout(0.3),\n            nn.Linear(64, num_classes),\n            nn.Sigmoid(),\n        )\n\n    def forward(self, x):\n        B = x.shape[0]\n        h = self.stem(x)          # (B, 64,  2048)\n        h = self.blocks(h)        # (B, 256, 256)\n\n        # Temporal attention pooling\n        attn_w = self.temporal_attn(h)   # (B, 1, 256)\n        attn_w = F.softmax(attn_w, dim=-1)\n        h_pool = (h * attn_w).sum(dim=-1)  # (B, 256)\n\n        if self.return_embeddings:\n            return self.classifier(h_pool), h_pool\n        return self.classifier(h_pool)\n\n\n\n#   CELL 6 — MODEL 4: CrossDetectorGNN\n#\n#  THE PHYSICS MOTIVATION\n#  ─────────────────────\n#  A real gravitational wave MUST arrive at all three LIGO/Virgo\n#  detectors within 26 milliseconds (the light-travel time across\n#  the detector triangle). A detector glitch — a local disturbance\n#  like scattered light or seismic noise — appears in ONE detector\n#  only.\n#\n#  Current approach (in GravWaveFormer): stack all 3 detector\n#  channels as an RGB image. The model implicitly learns cross-\n#  detector consistency but it is never explicitly modelled.\n#\n#  CrossDetectorGNN makes it EXPLICIT:\n#  • Each detector is a NODE in a fully connected graph\n#  • Edges encode the cross-correlation between detector pairs\n#  • GNN message passing propagates signals between nodes\n#  • After 2 rounds of message passing, the graph readout\n#    captures whether signals are consistently coherent\n#\n#  THE GRAPH STRUCTURE\n#  ───────────────────\n#  Nodes:  {H1 (Hanford), L1 (Livingston), V1 (Virgo)} → 3 nodes\n#  Edges:  fully connected → 3 × 2 = 6 directed edges\n#  Node features: per-detector feature vector extracted from\n#    a small 1D CNN (128-dim embedding per detector)\n#  Edge features: normalised cross-correlation between each pair\n#    (computed from raw waveform segments)\n#\n#  GNN ARCHITECTURE: GraphSAGE\n#  ────────────────────────────\n#  GraphSAGE (Hamilton et al., 2017) aggregates neighbour features\n#  via mean pooling, then concatenates with the node\'s own features.\n#  It is inductive (generalises to new graph structures) and simple\n#  enough to train in minutes on 3-node graphs.\n#\n#  MESSAGE PASSING EQUATION:\n#  h_v^(k) = W × CONCAT(h_v^(k-1), MEAN_{u ∈ N(v)} h_u^(k-1))\n#  where v is a node, N(v) its neighbours, k the layer index.\n\ntry:\n    from torch_geometric.nn import SAGEConv, global_mean_pool\n    from torch_geometric.data import Data, Batch\n    GEO_AVAILABLE = True\n    print("✓ torch_geometric available — CrossDetectorGNN will use SAGEConv")\nexcept ImportError:\n    GEO_AVAILABLE = False\n    print("⚠ torch_geometric not found — using manual GNN fallback")\n\n\nclass DetectorEncoder(nn.Module):\n    """\n    Encodes a single detector\'s 1D waveform into a 128-dim feature vector.\n    Shared weights across all 3 detectors — physical symmetry assumption.\n    """\n    def __init__(self, in_len=4096, out_dim=128):\n        super().__init__()\n        self.encoder = nn.Sequential(\n            nn.Conv1d(1, 32, kernel_size=15, padding=7, bias=False),\n            nn.BatchNorm1d(32), nn.GELU(),\n            nn.MaxPool1d(4),          # 4096 → 1024\n            nn.Conv1d(32, 64, kernel_size=7, padding=3, bias=False),\n            nn.BatchNorm1d(64), nn.GELU(),\n            nn.MaxPool1d(4),          # 1024 → 256\n            nn.Conv1d(64, 128, kernel_size=5, padding=2, bias=False),\n            nn.BatchNorm1d(128), nn.GELU(),\n            nn.AdaptiveAvgPool1d(1),  # → 128\n        )\n        self.proj = nn.Linear(128, out_dim)\n\n    def forward(self, x):\n        # x: (B, time_len) — single detector waveform\n        h = self.encoder(x.unsqueeze(1))  # (B, 128, 1)\n        h = h.squeeze(-1)                 # (B, 128)\n        return self.proj(h)               # (B, out_dim)\n\n\ndef compute_cross_correlation(x, max_lag=200):\n    """\n    Compute normalised cross-correlation between every pair of detectors.\n\n    Parameters:\n        x : (B, 3, T) — batch of 3-detector waveforms\n        max_lag : int — max lag in samples to consider (±max_lag)\n\n    Returns:\n        corr_feats : (B, 3, 2*max_lag+1) cross-correlation per pair\n        max_corr   : (B, 3) max correlation coefficient per pair\n\n    Pairs: (H1-L1, H1-V1, L1-V1) → 3 pairs\n    """\n    B, C, T = x.shape\n    pairs = [(0, 1), (0, 2), (1, 2)]\n    max_corrs = []\n\n    for (i, j) in pairs:\n        # Normalise each channel\n        xi = x[:, i, :]                             # (B, T)\n        xj = x[:, j, :]\n        xi = (xi - xi.mean(dim=1, keepdim=True)) / (xi.std(dim=1, keepdim=True) + 1e-8)\n        xj = (xj - xj.mean(dim=1, keepdim=True)) / (xj.std(dim=1, keepdim=True) + 1e-8)\n\n        # Cross-correlation via FFT (O(T log T) vs O(T²))\n        Fx = torch.fft.rfft(xi, n=2*T)\n        Fy = torch.fft.rfft(xj, n=2*T)\n        cc = torch.fft.irfft(Fx * Fy.conj(), n=2*T)[:, :T] / T\n        max_corr = cc.abs().max(dim=1).values          # (B,)\n        max_corrs.append(max_corr)\n\n    return torch.stack(max_corrs, dim=1)               # (B, 3)\n\n\nclass CrossDetectorGNN(nn.Module):\n    """\n    Graph Neural Network over the 3 LIGO/Virgo detectors.\n\n    Pipeline:\n        Input: (B, 3, T) raw waveforms — 3 detectors × T time steps\n            → DetectorEncoder (shared weights)     → (B, 3, 128) node feats\n            → compute_cross_correlation            → (B, 3) edge feats\n            → 2× GraphSAGE message passing         → (B, 3, 128) updated nodes\n            → Global mean pool + edge feat concat  → (B, 256)\n            → Classifier head                      → P(GW)\n\n    Edge features (cross-correlations) modulate message weights:\n        higher correlation between two detectors → stronger message\n\n    If torch_geometric is unavailable, falls back to a manual\n    mean-pool GNN implementation that is functionally equivalent.\n    """\n    def __init__(self, node_dim=128, hidden_dim=128, num_classes=1,\n                 dropout=0.1, return_embeddings=False):\n        super().__init__()\n        self.return_embeddings = return_embeddings\n\n        # Shared encoder for all 3 detectors\n        self.detector_enc = DetectorEncoder(out_dim=node_dim)\n\n        # Edge feature projection (3 pair correlations → 32-dim edge embedding)\n        self.edge_proj = nn.Linear(3, 32)\n\n        if GEO_AVAILABLE:\n            # GraphSAGE layers\n            self.conv1 = SAGEConv(node_dim, hidden_dim)\n            self.conv2 = SAGEConv(hidden_dim, hidden_dim)\n        else:\n            # Fallback: manual mean-pool message passing\n            self.agg1 = nn.Linear(node_dim * 2, hidden_dim)\n            self.agg2 = nn.Linear(hidden_dim * 2, hidden_dim)\n\n        # Classifier takes concatenated: graph readout + edge features\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(hidden_dim + 32),\n            nn.Linear(hidden_dim + 32, 64),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(64, num_classes),\n            nn.Sigmoid(),\n        )\n\n        # Fully connected edge index for 3 nodes (used by torch_geometric)\n        self.register_buffer(\n            \'edge_index\',\n            torch.tensor([[0,0,1,1,2,2],\n                          [1,2,0,2,0,1]], dtype=torch.long)\n        )\n\n    def forward(self, x):\n        """\n        x: (B, 3, T) — batch of 3-detector waveforms\n        """\n        B = x.shape[0]\n\n        # 1. Encode each detector independently\n        node_feats = []\n        for d in range(3):\n            node_feats.append(self.detector_enc(x[:, d, :]))  # (B, node_dim)\n        node_feats = torch.stack(node_feats, dim=1)            # (B, 3, node_dim)\n\n        # 2. Compute edge features (cross-correlations)\n        edge_feats = compute_cross_correlation(x)              # (B, 3)\n        edge_emb   = F.gelu(self.edge_proj(edge_feats))       # (B, 32)\n\n        if GEO_AVAILABLE:\n            # 3. Build a batch of graphs for torch_geometric\n            # Flatten (B, 3, node_dim) to (B*3, node_dim)\n            node_flat = node_feats.view(B * 3, -1)            # (B*3, node_dim)\n\n            # Offset edge index for each graph in the batch\n            batch_vec   = torch.arange(B, device=x.device).repeat_interleave(3)\n            edge_batched = torch.cat([\n                self.edge_index + 3 * i for i in range(B)\n            ], dim=1)                                          # (2, B*6)\n\n            # Message passing\n            h = F.gelu(self.conv1(node_flat, edge_batched))   # (B*3, hidden)\n            h = F.gelu(self.conv2(h, edge_batched))           # (B*3, hidden)\n            h = global_mean_pool(h, batch_vec)                 # (B, hidden)\n\n        else:\n            # Manual mean-pool GNN fallback\n            # Layer 1: for each node, aggregate mean of neighbours\n            h = node_feats                                     # (B, 3, node_dim)\n            for _ in range(2):\n                agg = h.mean(dim=1, keepdim=True).expand_as(h)  # mean of all nodes\n                cat = torch.cat([h, agg], dim=-1)\n                if h.shape[-1] == node_feats.shape[-1]:\n                    h = F.gelu(self.agg1(cat))\n                else:\n                    h = F.gelu(self.agg2(cat))\n            h = h.mean(dim=1)                                  # (B, hidden)\n\n        # 4. Concatenate graph embedding + edge features → classify\n        combined = torch.cat([h, edge_emb], dim=-1)           # (B, hidden+32)\n        if self.return_embeddings:\n            return self.classifier(combined), combined\n        return self.classifier(combined)\n\n\n\n#  CELL 7 — MODEL 5: FlamingoBLIP2 VLM Explainer (NEW)\n#\n#  WHAT IS BLIP-2?\n#  ───────────────\n#  BLIP-2 (Bootstrapping Language-Image Pre-training, Li et al. 2023)\n#  is a vision-language model that connects a frozen image encoder\n#  (EVA-CLIP) and a frozen large language model (OPT or FlanT5)\n#  via a lightweight Q-Former bridge module.\n#\n#  It is the most accessible approximation to Flamingo (DeepMind)\n#  that can run on a Colab A100 using 8-bit quantization.\n#\n#  HOW WE USE IT:\n#  ─────────────\n#  BLIP-2 is a POST-HOC EXPLAINER, not a classifier.\n#  After GravWaveEnsemble makes a binary prediction, we pass the\n#  spectrogram image to BLIP-2 with a structured text prompt and\n#  get a natural language explanation of what the model "sees".\n#\n#  EXAMPLE PROMPTS AND EXPECTED OUTPUTS:\n#  ────────────────────────────────────────\n#  Prompt:  "Describe the pattern in this LIGO spectrogram image.\n#             Does it show a gravitational wave signal?"\n#  GW:      "The image shows a sweeping curve rising from low to high\n#             frequency, consistent with two massive objects spiralling\n#             together and merging — a gravitational wave chirp."\n#  Noise:   "The image shows irregular broadband noise with no coherent\n#             frequency structure, consistent with detector background."\n#\n#  This output appears in the Streamlit app and Notebook 5 evaluation.\n#\n#  RUNTIME NOTE:\n#  ─────────────\n#  BLIP-2 requires ~8GB GPU RAM with 8-bit quantization.\n#  Use Colab A100 (High-RAM) runtime for BLIP-2 inference.\n#  T4 (free tier) can run BLIP-2 with load_in_8bit=True but slowly.\n\nfrom transformers import Blip2Processor, Blip2ForConditionalGeneration\nfrom PIL import Image\nimport torchvision.transforms as T\n\nclass FlamingoBLIP2Explainer:\n    """\n    BLIP-2 Vision-Language Model for spectrogram explanation.\n\n    This is NOT a PyTorch nn.Module — it wraps a HuggingFace\n    pipeline and is used ONLY for inference, not training.\n\n    Usage:\n        explainer = FlamingoBLIP2Explainer()\n        explainer.load()\n        text = explainer.explain(spectrogram_tensor, prediction_prob)\n        print(text)\n    """\n\n    # Prompts for different prediction contexts\n    PROMPTS = {\n        \'high_confidence_gw\': (\n            "Question: This is a LIGO gravitational wave detector spectrogram "\n            "showing frequency (y-axis) vs time (x-axis). The detector found "\n            "this signal highly likely to be a gravitational wave event. "\n            "Describe the key visual features that indicate a gravitational wave chirp. "\n            "Answer:"\n        ),\n        \'high_confidence_noise\': (\n            "Question: This is a LIGO gravitational wave detector spectrogram "\n            "showing frequency (y-axis) vs time (x-axis). The detector classified "\n            "this as background noise with no gravitational wave present. "\n            "Describe the visual characteristics that indicate random noise. "\n            "Answer:"\n        ),\n        \'uncertain\': (\n            "Question: This is a LIGO gravitational wave detector spectrogram "\n            "showing frequency (y-axis) vs time (x-axis). The detector is uncertain "\n            "whether this contains a gravitational wave. "\n            "Describe what you see and any ambiguous features. "\n            "Answer:"\n        ),\n        \'generic\': (\n            "Question: Describe the most prominent visual pattern in this "\n            "time-frequency spectrogram from the LIGO gravitational wave detector. "\n            "Answer:"\n        ),\n    }\n\n    def __init__(self, model_name=\'Salesforce/blip2-opt-2.7b\'):\n        """\n        model_name options:\n        • \'Salesforce/blip2-opt-2.7b\'     ← 2.7B LLM, fits on T4 (8-bit)\n        • \'Salesforce/blip2-opt-6.7b\'     ← 6.7B LLM, needs A100\n        • \'Salesforce/blip2-flan-t5-xl\'   ← FlanT5 backbone, good at Q&A\n        """\n        self.model_name = model_name\n        self.model      = None\n        self.processor  = None\n        self.loaded     = False\n\n        # Inverse normalisation transform to convert spectrogram\n        # tensor back to a displayable PIL image for BLIP-2\n        self.to_pil = T.Compose([\n            T.Normalize(\n                mean=[-m/s for m, s in zip([0.485,0.456,0.406],\n                                           [0.229,0.224,0.225])],\n                std=[1/s for s in [0.229, 0.224, 0.225]]\n            ),\n            T.ToPILImage(),\n        ])\n\n    def load(self, load_in_8bit=True):\n        """\n        Load BLIP-2 model and processor from HuggingFace Hub.\n        Call this ONCE before any inference — it loads ~5-12 GB.\n\n        load_in_8bit=True: quantises to 8-bit, halving memory usage.\n        Set False on A100 for faster inference with full precision.\n        """\n        print(f"Loading BLIP-2 ({self.model_name})...")\n        print("  This downloads ~5-12 GB on first run (cached thereafter)")\n\n        self.processor = Blip2Processor.from_pretrained(self.model_name)\n        self.model = Blip2ForConditionalGeneration.from_pretrained(\n            self.model_name,\n            load_in_8bit = load_in_8bit,\n            device_map   = \'auto\',         # Automatically assigns layers to GPU/CPU\n            torch_dtype  = torch.float16,\n        )\n        self.model.eval()\n        self.loaded = True\n        print(f"✓ BLIP-2 loaded ({self.model_name})")\n\n    def _select_prompt(self, prob):\n        """Choose a prompt based on the model\'s confidence."""\n        if prob > 0.80:\n            return self.PROMPTS[\'high_confidence_gw\']\n        elif prob < 0.20:\n            return self.PROMPTS[\'high_confidence_noise\']\n        elif 0.35 < prob < 0.65:\n            return self.PROMPTS[\'uncertain\']\n        else:\n            return self.PROMPTS[\'generic\']\n\n    @torch.no_grad()\n    def explain(self, spectrogram_tensor, prediction_prob, max_new_tokens=120):\n        """\n        Generate a natural language explanation for one spectrogram.\n\n        Parameters:\n            spectrogram_tensor : (3, 224, 224) float tensor\n            prediction_prob    : float — P(GW) from the ensemble model\n            max_new_tokens     : int — max tokens to generate\n\n        Returns:\n            explanation : str — natural language description\n        """\n        if not self.loaded:\n            raise RuntimeError("Call .load() before .explain()")\n\n        # Convert tensor to PIL Image\n        img    = self.to_pil(spectrogram_tensor.cpu().clamp(0, 1))\n        prompt = self._select_prompt(prediction_prob)\n\n        # Tokenise and run inference\n        inputs = self.processor(\n            images  = img,\n            text    = prompt,\n            return_tensors = \'pt\',\n        ).to(next(self.model.parameters()).device)\n\n        output_ids  = self.model.generate(\n            **inputs,\n            max_new_tokens = max_new_tokens,\n            num_beams      = 4,\n            temperature    = 0.7,\n            repetition_penalty = 1.2,\n        )\n        explanation = self.processor.decode(\n            output_ids[0], skip_special_tokens=True\n        ).strip()\n\n        # Strip the prompt prefix if it was echoed back\n        if prompt.rstrip() in explanation:\n            explanation = explanation.replace(prompt.rstrip(), \'\').strip()\n\n        return explanation\n\n    @torch.no_grad()\n    def explain_batch(self, spectrogram_tensors, prediction_probs, max_new_tokens=120):\n        """\n        Generate explanations for a batch of spectrograms.\n\n        Parameters:\n            spectrogram_tensors : list of (3, 224, 224) tensors\n            prediction_probs    : list of float probabilities\n\n        Returns:\n            explanations : list of str\n        """\n        if not self.loaded:\n            raise RuntimeError("Call .load() before .explain_batch()")\n\n        explanations = []\n        for i, (tensor, prob) in enumerate(zip(spectrogram_tensors, prediction_probs)):\n            print(f"  Explaining sample {i+1}/{len(spectrogram_tensors)}...", end=\'\\r\')\n            explanation = self.explain(tensor, float(prob), max_new_tokens)\n            explanations.append(explanation)\n        print()\n        return explanations\n\n\n\n#   CELL 8 — ENSEMBLE: GravWaveEnsemble\n#\n#  ENSEMBLE STRATEGY: MLP META-LEARNER (STACKING)\n#  ───────────────────────────────────────────────\n#  Each individual model produces a probability P(GW) ∈ [0, 1].\n#  The ensemble combines them in two ways:\n#\n#  Method A — Simple averaging (no learnable params):\n#    P_ensemble = mean(P_gravwave, P_clip, P_wave1d, P_gnn)\n#    Fast, robust, no risk of overfitting the ensemble.\n#    Good for reporting a quick combined metric.\n#\n#  Method B — MLP meta-learner (stacking):\n#    The four probabilities AND their intermediate embeddings\n#    are concatenated and fed to a 2-layer MLP trained to\n#    optimally weight each model\'s contribution.\n#    Typically +1–3% AUC over simple averaging.\n#\n#  TRAINING STRATEGY:\n#  ──────────────────\n#  1. Train each component model independently (Notebook 4)\n#  2. Generate embedding predictions on a validation hold-out set\n#  3. Train the MLP meta-learner on those embeddings (20 epochs)\n#  4. The full forward pass (all 4 models + MLP) is the final system\n#\n#  EMBEDDING DIMENSIONS USED:\n#    GravWaveFormer CLS :  512\n#    CLIPWaveFormer CLS :  512\n#    WaveCNN1D pool     :  256\n#    CrossDetectorGNN   :  128 + 32 = 160\n#    ─────────────────────────\n#    Total              : 1440\n\nclass GravWaveEnsemble(nn.Module):\n    """\n    Multi-model ensemble for gravitational wave detection.\n\n    Combines GravWaveFormer, CLIPWaveFormer, WaveCNN1D, and\n    CrossDetectorGNN via a trained MLP meta-learner.\n\n    BLIP-2 is handled separately (inference only, not part of\n    the differentiable computation graph).\n\n    Forward pass requires:\n        spec_2d   : (B, 3, 224, 224) — spectrogram images\n        wave_1d   : (B, 3, 4096)     — raw 1D waveforms\n\n    Both representations come from the same LIGO window.\n    See Notebook 2 (preprocessing) for how they are saved.\n    """\n    def __init__(\n        self,\n        gravwave_ckpt = None,\n        clip_ckpt     = None,\n        wave1d_ckpt   = None,\n        gnn_ckpt      = None,\n        ensemble_mode = \'mlp\',      # \'mlp\' or \'average\'\n    ):\n        super().__init__()\n        self.ensemble_mode = ensemble_mode\n\n        # ── COMPONENT MODELS (with embedding return enabled) ──────\n        self.gravwave = GravWaveFormer(return_embeddings=True)\n        self.clip     = CLIPWaveFormer(return_embeddings=True)\n        self.wave1d   = WaveCNN1D(return_embeddings=True)\n        self.gnn      = CrossDetectorGNN(return_embeddings=True)\n\n        # Load pretrained weights if provided\n        if gravwave_ckpt:\n            self.gravwave.load_state_dict(\n                torch.load(gravwave_ckpt, map_location=\'cpu\')[\'model_state_dict\'])\n        if clip_ckpt:\n            self.clip.load_state_dict(\n                torch.load(clip_ckpt, map_location=\'cpu\')[\'model_state_dict\'])\n        if wave1d_ckpt:\n            self.wave1d.load_state_dict(\n                torch.load(wave1d_ckpt, map_location=\'cpu\')[\'model_state_dict\'])\n        if gnn_ckpt:\n            self.gnn.load_state_dict(\n                torch.load(gnn_ckpt, map_location=\'cpu\')[\'model_state_dict\'])\n\n        # ── MLP META-LEARNER ──────────────────────────────────────\n        # Input: 4 raw probabilities (4) + concatenated embeddings (1440)\n        # Total input size: 4 + 512 + 512 + 256 + 160 = 1444\n        EMB_DIM = 4 + 512 + 512 + 256 + 160\n\n        self.meta_learner = nn.Sequential(\n            nn.LayerNorm(EMB_DIM),\n            nn.Linear(EMB_DIM, 256),\n            nn.GELU(),\n            nn.Dropout(0.3),\n            nn.Linear(256, 64),\n            nn.GELU(),\n            nn.Dropout(0.2),\n            nn.Linear(64, 1),\n            nn.Sigmoid(),\n        )\n\n        # ── ATTENTION WEIGHTS (for interpretability) ──────────────\n        # Learnable weights that assign an importance score to each\n        # component model. Softmax-normalised → sum to 1.\n        # Used in ensemble_mode=\'weighted_average\'.\n        self.model_weights = nn.Parameter(torch.ones(4))\n\n    def forward(self, spec_2d, wave_1d):\n        """\n        Full ensemble forward pass.\n\n        Parameters:\n            spec_2d : (B, 3, 224, 224)\n            wave_1d : (B, 3, 4096)\n\n        Returns:\n            prob    : (B, 1) final ensemble probability\n            details : dict with per-model probabilities (for analysis)\n        """\n        # ── INDIVIDUAL MODEL PREDICTIONS ─────────────────────────\n        p_gw,  e_gw  = self.gravwave(spec_2d)    # (B,1), (B,512)\n        p_clip, e_clip = self.clip(spec_2d)       # (B,1), (B,512)\n        p_1d,  e_1d  = self.wave1d(wave_1d)      # (B,1), (B,256)\n        p_gnn, e_gnn = self.gnn(wave_1d)          # (B,1), (B,160)\n\n        details = {\n            \'p_gravwave\': p_gw.squeeze(1),\n            \'p_clip\':     p_clip.squeeze(1),\n            \'p_wave1d\':   p_1d.squeeze(1),\n            \'p_gnn\':      p_gnn.squeeze(1),\n        }\n\n        if self.ensemble_mode == \'average\':\n            # Simple unweighted average\n            prob = (p_gw + p_clip + p_1d + p_gnn) / 4\n            return prob, details\n\n        elif self.ensemble_mode == \'weighted_average\':\n            # Softmax-weighted average (weights are learnable)\n            w    = F.softmax(self.model_weights, dim=0)\n            probs = torch.cat([p_gw, p_clip, p_1d, p_gnn], dim=1)   # (B, 4)\n            prob  = (probs * w).sum(dim=1, keepdim=True)\n            details[\'model_weights\'] = w.detach()\n            return prob, details\n\n        else:\n            # \'mlp\' — MLP meta-learner on probs + embeddings\n            raw_probs = torch.cat([p_gw, p_clip, p_1d, p_gnn], dim=1)  # (B, 4)\n            embeddings = torch.cat([e_gw, e_clip, e_1d, e_gnn], dim=1) # (B, 1440)\n            combined   = torch.cat([raw_probs, embeddings], dim=1)      # (B, 1444)\n            prob       = self.meta_learner(combined)                     # (B, 1)\n            return prob, details\n\n    def freeze_backbones(self):\n        """\n        Freeze all component model weights.\n        Call this when training ONLY the meta-learner.\n        """\n        for m in [self.gravwave, self.clip, self.wave1d, self.gnn]:\n            for p in m.parameters():\n                p.requires_grad = False\n        print("✓ All backbone models frozen. Only meta-learner is trainable.")\n\n    def unfreeze_backbones(self):\n        """\n        Unfreeze all component model weights.\n        Call this for final end-to-end fine-tuning of the full system.\n        """\n        for m in [self.gravwave, self.clip, self.wave1d, self.gnn]:\n            for p in m.parameters():\n                p.requires_grad = True\n        print("✓ All backbone models unfrozen. Full system is trainable.")\n\n    def get_model_importances(self):\n        """\n        Returns a dict showing each model\'s current learned importance\n        (only meaningful for ensemble_mode=\'weighted_average\').\n        """\n        w = F.softmax(self.model_weights, dim=0).detach().cpu()\n        return {\n            \'GravWaveFormer\': float(w[0]),\n            \'CLIPWaveFormer\': float(w[1]),\n            \'WaveCNN1D\':      float(w[2]),\n            \'CrossDetectorGNN\': float(w[3]),\n        }\n\n\n\n\ndef build_model(name, return_embeddings=False, **kwargs):\n    builders = {\n        "gravwave": lambda: GravWaveFormer(return_embeddings=return_embeddings, **kwargs),\n        "clip":     lambda: CLIPWaveFormer(return_embeddings=return_embeddings, **kwargs),\n        "wave1d":   lambda: WaveCNN1D(return_embeddings=return_embeddings, **kwargs),\n        "gnn":      lambda: CrossDetectorGNN(return_embeddings=return_embeddings, **kwargs),\n        "ensemble": lambda: GravWaveEnsemble(**kwargs),\n    }\n    return builders[name]()\n'

with open(save_path, 'w') as f:
    f.write(file_content)

size = len(file_content)
print(f'✓ Saved: {save_path}')
print(f'  Size: {size:,} characters')

# Verify it loads and all classes are present
namespace = {}
exec(open(save_path).read(), namespace)
expected = ['DropPath', 'GravWaveFormer', 'CLIPWaveFormer',
            'DilatedResBlock1D', 'WaveCNN1D', 'DetectorEncoder',
            'CrossDetectorGNN', 'FlamingoBLIP2Explainer', 'GravWaveEnsemble',
            'compute_cross_correlation', 'build_model']
found    = [c for c in expected if c in namespace]
missing  = [c for c in expected if c not in namespace]

print(f'  Classes found   : {found}')
if missing:
    print(f'  ❌ Missing: {missing}')
else:
    print(f'  ✓ All {len(found)} classes/functions verified successfully')
    print()
    print('NEXT: Notebook_04_Training.ipynb')


✓ Saved: /content/drive/MyDrive/GravWaveFormer/gravwave_models.py
  Size: 42,596 characters
⚠ torch_geometric not found — using manual GNN fallback
  Classes found   : ['DropPath', 'GravWaveFormer', 'CLIPWaveFormer', 'DilatedResBlock1D', 'WaveCNN1D', 'DetectorEncoder', 'CrossDetectorGNN', 'FlamingoBLIP2Explainer', 'GravWaveEnsemble', 'compute_cross_correlation', 'build_model']
  ✓ All 11 classes/functions verified successfully

NEXT: Notebook_04_Training.ipynb
